# 股票資料查詢系統

這個筆記本提供了查詢特定日期、時間點和股票數值的功能。

In [ ]:
import pandas as pd
import os
from datetime import datetime
import glob

## 基本查詢功能

In [ ]:
# 讀取特定日期的資料
def load_data_for_date(date):
    """讀取特定日期的資料
    
    Args:
        date: 日期字串，格式 'YYYYMMDD'
    """
    base_path = r'D:\03_預估量相關資量\tw_kbar_1m_vol_exp'
    file_path = os.path.join(base_path, f'vol_exp_{date}.parquet')
    
    if os.path.exists(file_path):
        df = pd.read_parquet(file_path)
        print(f"成功讀取 {date} 的資料")
        print(f"資料形狀: {df.shape}")
        return df
    else:
        print(f"找不到檔案: {file_path}")
        return None

# 測試讀取
df = load_data_for_date('20250901')
if df is not None:
    df.head()

## 查詢單一股票在特定時間的數值

In [ ]:
def query_stock_at_time(df, stock_code, time):
    """查詢特定股票在特定時間的數值
    
    Args:
        df: DataFrame
        stock_code: 股票代碼 (如 '2330')
        time: 時間 (如 '0930')
    """
    stock_code = str(stock_code)
    
    # 假設 k_time 是索引
    if df.index.name == 'k_time':
        if time in df.index and stock_code in df.columns:
            value = df.loc[time, stock_code]
            print(f"股票 {stock_code} 在 {time} 的數值: {value}")
            return value
    else:
        print(f"找不到資料")
        return None

# 查詢範例
if df is not None:
    # 查詢台積電 (2330) 在 09:30 的數值
    value = query_stock_at_time(df, '2330', '0930')

## 查詢多個時間點

In [ ]:
def query_stock_multiple_times(df, stock_code, times):
    """查詢股票在多個時間點的數值
    
    Args:
        df: DataFrame
        stock_code: 股票代碼
        times: 時間列表
    """
    stock_code = str(stock_code)
    
    if stock_code not in df.columns:
        print(f"找不到股票代碼: {stock_code}")
        return None
    
    # 篩選指定時間
    if df.index.name == 'k_time':
        result = df.loc[df.index.isin(times), stock_code]
        return result
    
# 查詢範例
if df is not None:
    times = ['0900', '0930', '1000', '1100', '1300']
    result = query_stock_multiple_times(df, '2330', times)
    if result is not None:
        print("台積電在各時間點的數值:")
        print(result)

## 互動式查詢介面

In [ ]:
def interactive_query():
    """互動式查詢介面"""
    # 輸入參數
    date = input("請輸入日期 (YYYYMMDD): ")
    stock_code = input("請輸入股票代碼: ")
    time = input("請輸入時間 (HHMM): ")
    
    # 載入資料
    df = load_data_for_date(date)
    if df is not None:
        # 查詢數值
        value = query_stock_at_time(df, stock_code, time)
        return value
    return None

# 執行互動式查詢
# interactive_query()

## 批次查詢多個日期

In [ ]:
def batch_query(dates, stock_code, time):
    """批次查詢多個日期的相同股票和時間
    
    Args:
        dates: 日期列表
        stock_code: 股票代碼
        time: 時間
    """
    results = {}
    
    for date in dates:
        df = load_data_for_date(date)
        if df is not None:
            value = query_stock_at_time(df, stock_code, time)
            results[date] = value
    
    # 轉換為 Series 方便查看
    results_series = pd.Series(results)
    results_series.index.name = 'date'
    return results_series

# 批次查詢範例
# dates = ['20250901', '20250902', '20250903']
# results = batch_query(dates, '2330', '0930')
# print(results)

## 取得可用的日期和股票列表

In [ ]:
def get_available_dates():
    """取得所有可用的日期"""
    base_path = r'D:\03_預估量相關資量\tw_kbar_1m_vol_exp'
    pattern = os.path.join(base_path, 'vol_exp_*.parquet')
    files = glob.glob(pattern)
    
    dates = []
    for file in files:
        filename = os.path.basename(file)
        if filename.startswith('vol_exp_') and filename.endswith('.parquet'):
            date = filename.replace('vol_exp_', '').replace('.parquet', '')
            dates.append(date)
    
    dates.sort()
    return dates

# 顯示可用日期
available_dates = get_available_dates()
print(f"可用日期數量: {len(available_dates)}")
if available_dates:
    print(f"日期範圍: {available_dates[0]} 到 {available_dates[-1]}")
    print(f"前5個日期: {available_dates[:5]}")

## 繪製股票走勢圖

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  # 支援中文顯示

def plot_stock_trend(df, stock_code, title=None):
    """繪製股票單日走勢圖
    
    Args:
        df: DataFrame
        stock_code: 股票代碼
        title: 圖表標題
    """
    stock_code = str(stock_code)
    
    if stock_code not in df.columns:
        print(f"找不到股票代碼: {stock_code}")
        return
    
    # 取得資料
    if df.index.name == 'k_time':
        data = df[stock_code]
    else:
        print("資料格式不符")
        return
    
    # 繪圖
    plt.figure(figsize=(12, 6))
    plt.plot(data.index, data.values, marker='o', markersize=2)
    
    if title:
        plt.title(title)
    else:
        plt.title(f"股票 {stock_code} 走勢圖")
    
    plt.xlabel('時間')
    plt.ylabel('數值')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 繪圖範例
if df is not None:
    plot_stock_trend(df, '2330', '台積電(2330) 2025/09/01 走勢圖')

In [2]:
import pandas as pd 
df = pd.read_parquet(r'D:\feature_data\feature\2025-01-02\1216.parquet')
df.columns

Index(['type', 'time', 'price', 'volume', 'tick_type', 'vwap', 'vwap_5m',
       'day_high', 'ratio_15s_180s', 'ratio_15s_180s_w321', 'ratio_30s_180s',
       'ratio_30s_180s_w321', 'ratio_60s_180s', 'ratio_60s_180s_w321',
       'ratio_15s_300s', 'ratio_buy5min_54321', 'ratio_buy5min_33221',
       'ratio_buy5min', 'ratio_buy3min_321', 'ratio_buy3min', 'high_1m',
       'low_1m', 'high_2m', 'low_2m', 'high_3m', 'low_3m', 'high_5m', 'low_5m',
       'low_7m', 'low_10m', 'low_15m', 'pct_2min', 'pct_3min', 'pct_5min',
       'ratio_10s_30s', 'ratio_15s_60s', 'ratio_30s_120s', 'ratio_15s_45s',
       'ratio_15s_30s', 'vol_buy_15s', 'cum_buy_vol', 'cum_sell_vol',
       'cum_total_vol', 'bid_volume_5level', 'ask_volume_5level',
       'bid_ask_ratio', 'obi', 'bid1_volume', 'bid2_volume', 'bid3_volume',
       'bid4_volume', 'bid5_volume', 'ask1_volume', 'ask2_volume',
       'ask3_volume', 'ask4_volume', 'ask5_volume'],
      dtype='object')